Purpose: Clean, standardize, harmonize, and consolidate Customers and Products datasets from Company A and Company B into Silver Delta tables.

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *

In [0]:
base_path = "/Volumes/workspace/default/fmcg_data/"
bronze_path = base_path + "bronze/"
silver_path= "/Volumes/workspace/default/fmcg_data/silver/"

In [0]:
customers_a = spark.read.format("delta").load(bronze_path + "customers_a")
customers_b = spark.read.format("delta").load(bronze_path + "customers_b")

products_a = spark.read.format("delta").load(bronze_path + "products_a")
products_b = spark.read.format("delta").load(bronze_path + "products_b")


Cleaning customers_A


In [0]:
customers_a.printSchema()

#display(customers_a)

print("Rows:", customers_a.count())

root
 |-- Customer_ID: integer (nullable = true)
 |-- Customer_Name: string (nullable = true)
 |-- Email: string (nullable = true)
 |-- Phone: string (nullable = true)
 |-- Gender: string (nullable = true)
 |-- Date_of_Birth: date (nullable = true)
 |-- City: string (nullable = true)
 |-- State: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- Join_Date: date (nullable = true)
 |-- Loyalty_Status: string (nullable = true)
 |-- Created_At: timestamp (nullable = true)
 |-- Last_Updated: timestamp (nullable = true)

Rows: 510


In [0]:
customers_a.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in customers_a.columns
]).show()

+-----------+-------------+-----+-----+------+-------------+----+-----+-------+---------+--------------+----------+------------+
|Customer_ID|Customer_Name|Email|Phone|Gender|Date_of_Birth|City|State|Country|Join_Date|Loyalty_Status|Created_At|Last_Updated|
+-----------+-------------+-----+-----+------+-------------+----+-----+-------+---------+--------------+----------+------------+
|          0|            0|   25|    9|     0|            0|   0|    0|      0|        0|             0|         0|           0|
+-----------+-------------+-----+-----+------+-------------+----+-----+-------+---------+--------------+----------+------------+



In [0]:
# phones, emails are missing in few records

In [0]:
customers_a.groupBy("Customer_ID") \
    .count() \
    .filter(col("count") > 1) \
    .show()

+-----------+-----+
|Customer_ID|count|
+-----------+-----+
|       1041|    2|
|       1076|    2|
|       1449|    2|
|       1213|    2|
|       1475|    2|
|       1270|    2|
|       1242|    2|
|       1482|    2|
|       1223|    2|
|       1281|    2|
+-----------+-----+



In [0]:
customers_a = customers_a.dropDuplicates(["Customer_ID"])

customers_a.groupBy("Customer_ID") \
    .count() \
    .filter(col("count") > 1) \
    .show()

+-----------+-----+
|Customer_ID|count|
+-----------+-----+
+-----------+-----+



In [0]:
# Standardising
customers_a = customers_a.withColumn("State",initcap(col("State")))
customers_a = customers_a.withColumn("Loyalty_Status",initcap(col("Loyalty_Status")))
customers_a = customers_a.withColumn("Phone",
    regexp_replace(col("Phone"), "[^0-9]", "")
)
print("Rows:", customers_a.count())

Rows: 500


Cleaning customers_B

In [0]:
customers_b = spark.read.format("delta").load(bronze_path + "customers_b")

In [0]:
customers_b.printSchema()

#display(customers_b)

print("Rows:", customers_b.count())

root
 |-- Cust_ID: string (nullable = true)
 |-- Full_Name: string (nullable = true)
 |-- Email_Address: string (nullable = true)
 |-- Contact_No: string (nullable = true)
 |-- Town: string (nullable = true)
 |-- Membership: string (nullable = true)
 |-- Signup_Date: date (nullable = true)

Rows: 352


In [0]:
customers_b.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in customers_b.columns
]).show()

+-------+---------+-------------+----------+----+----------+-----------+
|Cust_ID|Full_Name|Email_Address|Contact_No|Town|Membership|Signup_Date|
+-------+---------+-------------+----------+----+----------+-----------+
|      0|        0|           23|        10|   0|         0|          0|
+-------+---------+-------------+----------+----+----------+-----------+



In [0]:
customers_b.groupBy("Cust_ID") \
    .count() \
    .filter(col("count") > 1) \
    .show()

+-------+-----+
|Cust_ID|count|
+-------+-----+
| CB1128|    2|
| CB1063|    2|
+-------+-----+



In [0]:
customers_b = customers_b.dropDuplicates(["Cust_ID"])

In [0]:
#standardising
customers_b = customers_b.withColumn("Town",initcap(col("Town")))
customers_b = customers_b.withColumn("Membership",initcap(col("Membership")))
customers_b = customers_b.withColumn("Contact_No",
    regexp_replace(col("Contact_No"), "[^0-9]", "")
)


In [0]:
print("Rows:", customers_b.count())

customers_b.groupBy("Cust_ID") \
    .count() \
    .filter(col("count") > 1) \
    .show()

# display(customers_b)

Rows: 350
+-------+-----+
|Cust_ID|count|
+-------+-----+
+-------+-----+



Harmonize Customers B Schema to match that of A, and then perform Union to create a single customers table

In [0]:
# matching the schema of B with A
# adding the missing columns

customers_b = customers_b \
    .withColumnRenamed("Cust_ID", "Customer_ID") \
    .withColumnRenamed("Full_Name", "Customer_Name") \
    .withColumnRenamed("Email_Address", "Email") \
    .withColumnRenamed("Contact_No", "Phone") \
    .withColumnRenamed("Town", "City") \
    .withColumnRenamed("Membership", "Loyalty_Status") \
    .withColumnRenamed("Signup_Date", "Join_Date")\
    .withColumn("Gender", lit(None).cast("string")) \
    .withColumn("Date_of_Birth", lit(None).cast("date")) \
    .withColumn("State", lit(None).cast("string")) \
    .withColumn("Country", lit(None).cast("string"))\
    .withColumn("Created_At", lit(None).cast("timestamp")) \
    .withColumn("Last_Updated", lit(None).cast("timestamp"))

In [0]:
customers_b = customers_b.select(customers_a.columns)

In [0]:
customers_a.printSchema()
customers_b.printSchema()

root
 |-- Customer_ID: integer (nullable = true)
 |-- Customer_Name: string (nullable = true)
 |-- Email: string (nullable = true)
 |-- Phone: string (nullable = true)
 |-- Gender: string (nullable = true)
 |-- Date_of_Birth: date (nullable = true)
 |-- City: string (nullable = true)
 |-- State: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- Join_Date: date (nullable = true)
 |-- Loyalty_Status: string (nullable = true)
 |-- Created_At: timestamp (nullable = true)
 |-- Last_Updated: timestamp (nullable = true)

root
 |-- Customer_ID: string (nullable = true)
 |-- Customer_Name: string (nullable = true)
 |-- Email: string (nullable = true)
 |-- Phone: string (nullable = true)
 |-- Gender: string (nullable = true)
 |-- Date_of_Birth: date (nullable = true)
 |-- City: string (nullable = true)
 |-- State: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- Join_Date: date (nullable = true)
 |-- Loyalty_Status: string (nullable = true)
 |-- Created_At:

In [0]:
customers_a = customers_a.withColumn(
    "Customer_ID",
    col("Customer_ID").cast("string")
)    # in table of A, customer_id is int so changing that before merging

In [0]:
silver_customers = customers_a.unionByName(customers_b)
print("Total Customers:", silver_customers.count())

Total Customers: 850


In [0]:
# saving merged dataframe 
silver_customers.write \
    .format("delta") \
    .mode("overwrite") \
    .save(silver_path + "customers")

CLEANING PRODUCTS_A

In [0]:
products_a.printSchema()

#display(products_a)

print("Rows:", products_a.count())

root
 |-- Product_ID: string (nullable = true)
 |-- Product_Name: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- Brand: string (nullable = true)
 |-- Unit_Price: integer (nullable = true)
 |-- Supplier: string (nullable = true)
 |-- Launch_Date: date (nullable = true)
 |-- Created_At: date (nullable = true)
 |-- Last_Updated: date (nullable = true)

Rows: 153


In [0]:
products_a.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in products_a.columns
]).show()

+----------+------------+--------+-----+----------+--------+-----------+----------+------------+
|Product_ID|Product_Name|Category|Brand|Unit_Price|Supplier|Launch_Date|Created_At|Last_Updated|
+----------+------------+--------+-----+----------+--------+-----------+----------+------------+
|         0|           0|       0|    0|         0|       8|          0|         0|           0|
+----------+------------+--------+-----+----------+--------+-----------+----------+------------+



In [0]:
products_a = products_a.dropDuplicates(["Product_ID"])

In [0]:
#standardise category,brand
products_a = products_a.withColumn("Category",initcap(col("Category")))
products_a = products_a.withColumn("Brand",initcap(col("Brand")))
products_a = products_a.withColumn(
    "Unit_Price",
    coalesce(col("Unit_Price"), lit(0))
)
products_a.printSchema()

root
 |-- Product_ID: string (nullable = true)
 |-- Product_Name: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- Brand: string (nullable = true)
 |-- Unit_Price: integer (nullable = false)
 |-- Supplier: string (nullable = true)
 |-- Launch_Date: date (nullable = true)
 |-- Created_At: date (nullable = true)
 |-- Last_Updated: date (nullable = true)



CLEANING PRODUCTS_B

In [0]:
products_b.printSchema()

root
 |-- Prod_ID: string (nullable = true)
 |-- Item_Name: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- Brand: string (nullable = true)
 |-- MRP: double (nullable = true)
 |-- Vendor: string (nullable = true)
 |-- Launch_Date: date (nullable = true)
 |-- Created_On: date (nullable = true)
 |-- Updated_On: date (nullable = true)



In [0]:
#display(products_b)

print("Rows:", products_b.count())

Rows: 123


In [0]:
products_b.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in products_b.columns
]).show()

+-------+---------+--------+-----+---+------+-----------+----------+----------+
|Prod_ID|Item_Name|Category|Brand|MRP|Vendor|Launch_Date|Created_On|Updated_On|
+-------+---------+--------+-----+---+------+-----------+----------+----------+
|      0|        0|       0|    0|  0|     9|          0|         0|         0|
+-------+---------+--------+-----+---+------+-----------+----------+----------+



In [0]:
products_b = products_b.dropDuplicates(["Prod_ID"])

In [0]:
products_b = products_b.withColumn( "Category",initcap(col("Category")))
products_b = products_b.withColumn( "Brand",initcap(col("Brand")))

print("Rows:", products_b.count())

products_b.groupBy("Prod_ID") \
    .count() \
    .filter(col("count") > 1) \
    .show()


Rows: 120
+-------+-----+
|Prod_ID|count|
+-------+-----+
+-------+-----+



harmonizing products_a and products_b , and merging them 

In [0]:
# seeting unit_price as double 
products_a = products_a.withColumn(
    "Unit_Price",
    col("Unit_Price").cast("double")
)

In [0]:
# standardising columns
products_b = products_b \
    .withColumnRenamed("Prod_ID", "Product_ID") \
    .withColumnRenamed("Item_Name", "Product_Name") \
    .withColumnRenamed("MRP", "Unit_Price") \
    .withColumnRenamed("Vendor", "Supplier") \
    .withColumnRenamed("Created_On", "Created_At") \
    .withColumnRenamed("Updated_On", "Last_Updated")

In [0]:
products_b = products_b.withColumn(
    "Unit_Price",
    col("Unit_Price").cast("double")
)

In [0]:
# verifying
products_a.printSchema()
products_b.printSchema()

root
 |-- Product_ID: string (nullable = true)
 |-- Product_Name: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- Brand: string (nullable = true)
 |-- Unit_Price: double (nullable = false)
 |-- Supplier: string (nullable = true)
 |-- Launch_Date: date (nullable = true)
 |-- Created_At: date (nullable = true)
 |-- Last_Updated: date (nullable = true)

root
 |-- Product_ID: string (nullable = true)
 |-- Product_Name: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- Brand: string (nullable = true)
 |-- Unit_Price: double (nullable = true)
 |-- Supplier: string (nullable = true)
 |-- Launch_Date: date (nullable = true)
 |-- Created_At: date (nullable = true)
 |-- Last_Updated: date (nullable = true)



In [0]:
products_b = products_b.select(products_a.columns)
silver_products = products_a.unionByName(products_b)

In [0]:
# saving the merged dataframe
silver_products.write \
    .format("delta") \
    .mode("overwrite") \
    .save(silver_path + "products")

In [0]:
# verifying 
spark.read.format("delta") \
    .load("/Volumes/workspace/default/fmcg_data/silver/customers") \
    .count()

850